# 03 — Model Training

Trains and evaluates models to predict comment persuasiveness on CMV.
Uses a thread-grouped train/test split to prevent data leakage.

Loads directly from `results/cmv_comments_df.csv.zip` — no need to re-run notebooks 01 or 02.


## Load data


In [ ]:
import pandas as pd
import zipfile

with zipfile.ZipFile('../results/cmv_comments_df.csv.zip') as z:
    comments_df = pd.read_csv(z.open(z.namelist()[0]))

print(f'Loaded {len(comments_df)} rows, {comments_df.shape[1]} columns')
comments_df.head(3)


## Task 1.1 — Add `thread_id`

Each Reddit thread can have many comments. A naive train/test split by comment row
allows comments from the same thread to appear on both sides — the model sees the
thread's writing style and topic during training and effectively 'recognises' it at
test time. This inflates performance artificially.

To fix this we split by thread, keeping all comments from a thread on the same side.
`thread_id` is a stable identifier for each thread, derived by hashing the
`original_post` text — which is identical for every comment in the same thread.


In [ ]:
import hashlib

comments_df['thread_id'] = comments_df['original_post'].apply(
    lambda x: hashlib.md5(str(x).encode()).hexdigest()
)

print(f'Unique threads : {comments_df["thread_id"].nunique()}')
print(f'Total comments : {len(comments_df)}')
print(f'Avg comments per thread: {len(comments_df) / comments_df["thread_id"].nunique():.1f}')


## Task 1.2 — Split by thread, not by comment

Using `thread_id` as a grouping key, `GroupShuffleSplit` ensures every comment
from a given thread lands entirely in train or entirely in test — never both.


In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder

numeric_features = ['sentiment', 'adj_adv_ratio', 'fk_grade_level', 'fk_reading_ease', 'evidence_count',
                     'sentiment_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'fk_reading_ease_op', 'evidence_count_op']
label = 'is_convincing'

# Encode categorical tone features
categorical_features = ['tone_google', 'tone_google_op']
onehot_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features].fillna('unknown'))
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out(), index=comments_df.index)

comments_df['is_formal']    = (comments_df['style_label']    == 'formal').astype(int)
comments_df['is_formal_op'] = (comments_df['style_label_op'] == 'formal').astype(int)

X = pd.concat([tone_df,
               comments_df[['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + numeric_features]],
              axis=1)
y = comments_df[label]
groups = comments_df['thread_id']

print(f'Features : {X.shape[1]}')
print(f'Samples  : {X.shape[0]}')
print(f'Positive rate: {y.mean():.3f}')


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Assertion — zero thread overlap between train and test
train_threads = set(groups.iloc[train_idx])
test_threads  = set(groups.iloc[test_idx])
shared = train_threads & test_threads
assert len(shared) == 0, f'LEAKAGE: {len(shared)} shared threads!'
print(f'Shared threads between train and test: {len(shared)}  ✓')
print(f'Train: {len(X_train)} comments from {len(train_threads)} threads')
print(f'Test : {len(X_test)} comments from {len(test_threads)} threads')


## Task 1.3 — Preprocessing inside a Pipeline

Fitting a scaler on the full dataset before splitting causes the test set to
influence the scaling — a subtle form of leakage. Wrapping scaler + resampler +
model in an `imblearn.Pipeline` ensures preprocessing only ever sees training data.

We define three pipeline variants to compare:
- **No resampling** — trains on the naturally imbalanced data
- **Undersampling** (RandomUnderSampler) — reduces the majority class
- **Oversampling** (SMOTE) — generates synthetic minority-class samples


In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import xgboost as xgb
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

def make_pipelines(clf, resampling='none'):
    """Return an imblearn Pipeline with scaler, optional resampler, and classifier."""
    steps = [('scaler', RobustScaler())]
    if resampling == 'under':
        steps.append(('sampler', RandomUnderSampler(random_state=42)))
    elif resampling == 'over':
        steps.append(('sampler', SMOTE(random_state=42)))
    steps.append(('clf', clf))
    return Pipeline(steps)

# One pipeline per model × resampling strategy
pipelines = {
    'LogReg (none)':  make_pipelines(LogisticRegression(random_state=42, max_iter=1000), 'none'),
    'LogReg (under)': make_pipelines(LogisticRegression(random_state=42, max_iter=1000), 'under'),
    'LogReg (over)':  make_pipelines(LogisticRegression(random_state=42, max_iter=1000), 'over'),
    'RF (none)':      make_pipelines(RandomForestClassifier(random_state=42, n_estimators=100), 'none'),
    'RF (under)':     make_pipelines(RandomForestClassifier(random_state=42, n_estimators=100), 'under'),
    'RF (over)':      make_pipelines(RandomForestClassifier(random_state=42, n_estimators=100), 'over'),
    'XGB (none)':     make_pipelines(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'none'),
    'XGB (under)':    make_pipelines(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'under'),
    'XGB (over)':     make_pipelines(xgb.XGBClassifier(random_state=42, eval_metric='logloss'), 'over'),
}

print(f'{len(pipelines)} pipelines defined')
print('Each pipeline: RobustScaler → [optional resampler] → classifier')
print('Scaler and resampler fit only on training data — no leakage.')


# 9. Conclusion and Future Research

In this work we investigated comments from the CMV forum.

We created a new and rich dataset with comments, context (thread text by the author of the comment), original post and several features extracted from the comment + context text and from the original post text.

This dataset can be extended in the future by adding more features and can be used for further studies.

Possible features to add:
* Lexical Complexity:
  * Type-token ratio (TTR) – Measures vocabulary richness.
  * Word frequency analysis – Uses external corpora to determine whether a text contains rare or common words.
* Syntactic Complexity:
  * Parse tree depth – Longer, more nested sentences are syntactically complex.
  * Mean length of sentences (MLS) – Longer sentences generally indicate higher complexity.

The dataset can be downloaded from [here](https://drive.google.com/file/d/1cJWCp6StLol6M9yQHdI6irssBkTi-dKO/view?usp=sharing).

We trained multiple ML models to predict persuasiveness and addressed the class imbalance problem by experimenting with undersampling and oversampling (SMOTE) techniques.

Results are reported in the evaluation section above.


# 10. References

1. Priniski, J.H. & Horne, Z. (2018). [Attitude Change on Reddit's Change My View](https://jpriniski.github.io/papers/cogsci-reddit.pdf). In T.T. Rogers, M. Rau, X. Zhu, & C. W. Kalish (Eds.), Proceedings of the 40th Annual Conference of the Cognitive Science Society (pp. 2276-2281). Austin, TX: Cognitive Science Society.

2. [GitHub repo](https://github.com/jpriniski/CMV) of source [1]

3. Xiao, L., Mensah, H. (2022). [How Does the Thread Level of a Comment Affect its Perceived Persuasiveness? A Reddit Study](https://doi.org/10.1007/978-3-031-10464-0_55). In: Arai, K. (eds) Intelligent Computing. SAI 2022. Lecture Notes in Networks and Systems, vol 507. Springer, Cham.

4. Ivan Habernal and Iryna Gurevych. 2016. [What makes a convincing argument? Empirical analysis and detecting attributes of convincingness in Web argumentation](https://aclanthology.org/D16-1129.pdf). In Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing, pages 1214–1223, Austin, Texas. Association for Computational Linguistics.

## This Work Repo
https://github.com/jct-nlp/change-my-view-2025
